In [70]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [71]:
load_dotenv()
username = "root"
password = os.getenv("MYSQL_PASSWORD")
host = "localhost"
port = 3306
database = "food_delivery_db"
print("MySQL settings loaded")
print("Host:", host)
print("User:", username)
print("Database:", database)
print("Password loaded:", password is not None)

MySQL settings loaded
Host: localhost
User: root
Database: food_delivery_db
Password loaded: True


In [72]:
server_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}"
)
with server_engine.connect() as conn:
    conn.execute(text(
        f"CREATE DATABASE IF NOT EXISTS {database}"
    ))
    conn.commit()
print("Database created successfully!")

Database created successfully!


In [73]:
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)
try:
    with engine.connect() as conn:
        print("Connected to food_delivery_db successfully!")
except Exception as e:
    print("Connection failed:")
    print(e)

Connected to food_delivery_db successfully!


In [74]:
csv_path = r"C:\Users\admin\Desktop\project 2\cleaned_food_delivery.csv"
df_sql = pd.read_csv(csv_path)
print("Dataset loaded successfully!")
print("Rows:", df_sql.shape[0])
print("Columns:", df_sql.shape[1])

Dataset loaded successfully!
Rows: 100000
Columns: 30


In [75]:
df_sql.to_sql(
    "food_delivery_orders",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi"
)
print("Dataset uploaded successfully!")

Dataset uploaded successfully!


In [76]:
query = """
SELECT COUNT(*) AS total_rows
FROM food_delivery_orders;
"""
result = pd.read_sql(query, engine)
print(result)

   total_rows
0      100000


## top spenders

In [77]:
query = """
SELECT
    Customer_ID,
    SUM(Final_Amount) AS Total_Spending
FROM food_delivery_orders
GROUP BY Customer_ID
ORDER BY Total_Spending DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result

,Customer_ID,Total_Spending
0,CUST5267,51827.0
1,CUST1606,51171.0
2,CUST6706,47389.0
3,CUST6252,45925.0
4,CUST5534,45218.0
5,CUST1239,44983.0
6,CUST6457,44401.0
7,CUST4431,44335.0
8,CUST1968,44281.0
9,CUST6293,44211.0


## Age Group vs Average Order Value

In [78]:
query = """
SELECT
    Customer_Age_Group,
    AVG(Order_Value) AS Average_Order_Value
FROM food_delivery_orders
GROUP BY Customer_Age_Group
ORDER BY Average_Order_Value DESC;
"""
result = pd.read_sql(query, engine)
result

,Customer_Age_Group,Average_Order_Value
0,Adult,1806.888441
1,Senior,1784.602071
2,Middle Age,1784.536970
3,Young Adult,1782.305970


## Weekend vs Weekday

In [79]:
query = """
SELECT
    Order_Day_Type,
    COUNT(*) AS Total_Orders,
    AVG(Order_Value) AS Average_Order_Value,
    SUM(Final_Amount) AS Total_Revenue
FROM food_delivery_orders
GROUP BY Order_Day_Type
ORDER BY Total_Orders DESC;
"""
result = pd.read_sql(query, engine)
result

,Order_Day_Type,Total_Orders,Average_Order_Value,Total_Revenue
0,Weekday,71370,1789.416968,121578363.0
1,Weekend,28630,1780.775061,48516143.0


## Monthly Revenue

In [80]:
query = """
SELECT
    DATE_FORMAT(Order_Date, '%%Y-%%m') AS Month,
    SUM(Final_Amount) AS Total_Revenue
FROM food_delivery_orders
WHERE Order_Date IS NOT NULL
GROUP BY DATE_FORMAT(Order_Date, '%%Y-%%m')
ORDER BY Month;
"""
result = pd.read_sql(query, engine)
result


,Month,Total_Revenue
0,2024-01,14487752.0
1,2024-02,13405376.0
2,2024-03,14187318.0
3,2024-04,13773642.0
4,2024-05,14213329.0
5,2024-06,13959821.0
6,2024-07,14650673.0
7,2024-08,14054729.0
8,2024-09,14097100.0
9,2024-10,14007992.0


## Discount vs Profit Margin

In [81]:
query = """
SELECT
    COUNT(*) AS Total_Rows,
    SUM(Discount_Applied IS NULL) AS Missing_Discount,
    SUM(Order_Value IS NULL) AS Missing_Order_Value,
    SUM(Final_Amount IS NULL) AS Missing_Final_Amount,
    SUM(Delivery_Time_Min IS NULL) AS Missing_Delivery_Time,
    SUM(Distance_km IS NULL) AS Missing_Distance,
    SUM(Delivery_Rating IS NULL) AS Missing_Delivery_Rating
FROM food_delivery_orders;
"""
result = pd.read_sql(query, engine)
result

,Total_Rows,Missing_Discount,Missing_Order_Value,Missing_Final_Amount,Missing_Delivery_Time,Missing_Distance,Missing_Delivery_Rating
0,100000,0.0,0.0,0.0,0.0,0.0,0.0


In [82]:
query = """
SELECT
    Discount_Applied,
    AVG(Profit_Margin_Pct) AS Average_Profit_Margin_Pct
FROM food_delivery_orders
GROUP BY Discount_Applied
ORDER BY Discount_Applied;
"""

result = pd.read_sql(query, engine)

result

,Discount_Applied,Average_Profit_Margin_Pct
0,0.0,15.065465
1,20.0,14.946647
2,50.0,14.902559
3,100.0,15.132570
4,300.0,15.267771


## High-Revenue Cities

In [83]:
query = """
SELECT
    City,
    COUNT(*) AS Total_Orders,
    SUM(Final_Amount) AS Total_Revenue,
    AVG(Final_Amount) AS Average_Order_Value
FROM food_delivery_orders
GROUP BY City
ORDER BY Total_Revenue DESC;
"""
result = pd.read_sql(query, engine)
result

,City,Total_Orders,Total_Revenue,Average_Order_Value
0,Bangalore,16732,28651047.0,1712.350406
1,Hyderabad,16884,28536321.0,1690.139837
2,Unknown,16726,28497679.0,1703.795229
3,Delhi,16695,28257765.0,1692.588500
4,Chennai,16470,28139875.0,1708.553430
5,Mumbai,16493,28011819.0,1698.406536


## Cuisine Performance

In [84]:
query = """
SELECT
    Cuisine_Type,
    COUNT(*) AS Total_Orders,
    SUM(Final_Amount) AS Total_Revenue,
    AVG(Final_Amount) AS Average_Order_Value
FROM food_delivery_orders
GROUP BY Cuisine_Type
ORDER BY Total_Revenue DESC;
"""
result = pd.read_sql(query, engine)
result

,Cuisine_Type,Total_Orders,Total_Revenue,Average_Order_Value
0,Unknown,16885,28911997.0,1712.288836
1,Indian,16685,28515082.0,1709.024993
2,Mexican,16602,28339695.0,1707.004879
3,Chinese,16651,28327539.0,1701.251516
4,Arabian,16658,28050563.0,1683.909413
5,Italian,16519,27949630.0,1691.968642


## Average Delivery Time by City

In [85]:
query = """
SELECT
    City,
    COUNT(*) AS Total_Orders,
    AVG(Delivery_Time_Min) AS Average_Delivery_Time
FROM food_delivery_orders
GROUP BY City
ORDER BY Average_Delivery_Time DESC;
"""
result = pd.read_sql(query, engine)
result

,City,Total_Orders,Average_Delivery_Time
0,Mumbai,16493,125.937246
1,Unknown,16726,125.110307
2,Hyderabad,16884,125.031805
3,Delhi,16695,124.896376
4,Chennai,16470,124.549423
5,Bangalore,16732,124.373297


## Distance vs Delivery Delay

In [86]:
query = """
SELECT
    CASE
        WHEN Distance_km <= 5 THEN '0-5 km'
        WHEN Distance_km <= 10 THEN '6-10 km'
        WHEN Distance_km <= 20 THEN '11-20 km'
        WHEN Distance_km <= 30 THEN '21-30 km'
        ELSE '31-40 km'
    END AS Distance_Range,
    COUNT(*) AS Total_Orders,
    AVG(Delivery_Time_Min) AS Average_Delivery_Time
FROM food_delivery_orders
GROUP BY Distance_Range
ORDER BY
    CASE Distance_Range
        WHEN '0-5 km' THEN 1
        WHEN '6-10 km' THEN 2
        WHEN '11-20 km' THEN 3
        WHEN '21-30 km' THEN 4
        WHEN '31-40 km' THEN 5
    END;
"""
result = pd.read_sql(query, engine)
result

,Distance_Range,Total_Orders,Average_Delivery_Time
0,0-5 km,14857,124.475197
1,6-10 km,52001,124.923271
2,11-20 km,6646,125.517755
3,21-30 km,13333,125.475812
4,31-40 km,13163,125.015574


## Rating vs Delivery Time

In [87]:
query = """
SELECT
    Delivery_Rating,
    COUNT(*) AS Total_Orders,
    AVG(Delivery_Time_Min) AS Average_Delivery_Time
FROM food_delivery_orders
GROUP BY Delivery_Rating
ORDER BY Delivery_Rating;
"""
result = pd.read_sql(query, engine)
result

,Delivery_Rating,Total_Orders,Average_Delivery_Time
0,1.0,16818,124.969259
1,2.0,16896,124.912524
2,3.0,32931,124.462057
3,4.0,16885,125.654309
4,5.0,16470,125.416818


## Top-Rated Restaurants

In [88]:
query = """
SELECT
    Restaurant_Name,
    COUNT(*) AS Total_Orders,
    AVG(Restaurant_Rating) AS Average_Restaurant_Rating
FROM food_delivery_orders
GROUP BY Restaurant_Name
ORDER BY Average_Restaurant_Rating DESC
LIMIT 10;
"""

result = pd.read_sql(query, engine)

result

,Restaurant_Name,Total_Orders,Average_Restaurant_Rating
0,Restaurant_496,203,4.386207
1,Restaurant_101,178,4.384270
2,Restaurant_162,182,4.379121
3,Restaurant_355,181,4.371823
4,Restaurant_1,193,4.369430
5,Restaurant_481,190,4.362632
6,Restaurant_392,183,4.354645
7,Restaurant_352,194,4.354639
8,Restaurant_209,212,4.354245
9,Restaurant_119,176,4.351136


## Cancellation Rate by Restaurant

In [89]:
query = """
SELECT
    Restaurant_Name,
    COUNT(*) AS Total_Orders,
    SUM(CASE
        WHEN Order_Status = 'Cancelled' THEN 1
        ELSE 0
    END) AS Cancelled_Orders,
    ROUND(
        100.0 * SUM(CASE
            WHEN Order_Status = 'Cancelled' THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS Cancellation_Rate_Pct
FROM food_delivery_orders
GROUP BY Restaurant_Name
ORDER BY Cancellation_Rate_Pct DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)

result

,Restaurant_Name,Total_Orders,Cancelled_Orders,Cancellation_Rate_Pct
0,Restaurant_391,196,43.0,21.94
1,Restaurant_390,201,44.0,21.89
2,Restaurant_477,176,38.0,21.59
3,Restaurant_202,204,44.0,21.57
4,Restaurant_373,191,41.0,21.47
5,Restaurant_426,212,45.0,21.23
6,Restaurant_299,194,41.0,21.13
7,Restaurant_455,180,38.0,21.11
8,Restaurant_113,205,43.0,20.98
9,Restaurant_232,177,37.0,20.90


## Peak-Hour Demand

In [90]:
query = """
SELECT
    Peak_Hour_Indicator,
    COUNT(*) AS Total_Orders,
    SUM(Final_Amount) AS Total_Revenue,
    AVG(Final_Amount) AS Average_Order_Value
FROM food_delivery_orders
GROUP BY Peak_Hour_Indicator
ORDER BY Peak_Hour_Indicator;
"""
result = pd.read_sql(query, engine)
result

,Peak_Hour_Indicator,Total_Orders,Total_Revenue,Average_Order_Value
0,0,66547,113172200.0,1700.635641
1,1,33453,56922306.0,1701.560578


## Payment Mode

In [91]:
query = """
SELECT
    Payment_Mode,
    COUNT(*) AS Total_Orders,
    SUM(Final_Amount) AS Total_Revenue,
    AVG(Final_Amount) AS Average_Order_Value
FROM food_delivery_orders
GROUP BY Payment_Mode
ORDER BY Total_Orders DESC;
"""
result = pd.read_sql(query, engine)
result

,Payment_Mode,Total_Orders,Total_Revenue,Average_Order_Value
0,Card,20094,33952932.0,1689.704987
1,Wallet,20086,34056771.0,1695.547695
2,COD,19977,34232164.0,1713.578816
3,UPI,19932,34022519.0,1706.929510
4,Unknown,19911,33830120.0,1699.066847


# Cancellation Reason

In [92]:
query = """
SELECT
    Cancellation_Reason,
    COUNT(*) AS Total_Orders,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM food_delivery_orders),
        2
    ) AS Percentage_of_All_Orders
FROM food_delivery_orders
WHERE Order_Status = 'Cancelled'
GROUP BY Cancellation_Reason
ORDER BY Total_Orders DESC;
"""
result = pd.read_sql(query, engine)
result

,Cancellation_Reason,Total_Orders,Percentage_of_All_Orders
0,Unknown,6005,6.01
1,Late Delivery,3059,3.06
2,Customer Cancelled,2993,2.99
3,Restaurant Issue,2979,2.98


In [93]:
query = """
SELECT
    COUNT(*) AS Total_Orders,
    SUM(Final_Amount) AS Total_Revenue,
    AVG(Final_Amount) AS Average_Order_Value,
    AVG(Delivery_Time_Min) AS Average_Delivery_Time,
    100.0 * SUM(
        CASE WHEN Order_Status = 'Cancelled' THEN 1 ELSE 0 END
    ) / COUNT(*) AS Cancellation_Rate_Pct,
    AVG(Delivery_Rating) AS Average_Delivery_Rating,
    AVG(Profit_Margin_Pct) AS Profit_Margin_Pct
FROM food_delivery_orders;
"""
kpi_result = pd.read_sql(query, engine)
kpi_result

,Total_Orders,Total_Revenue,Average_Order_Value,Average_Delivery_Time,Cancellation_Rate_Pct,Average_Delivery_Rating,Profit_Margin_Pct
0,100000,170094506.0,1700.94506,124.98203,15.036,2.99293,15.0362


In [94]:
query = """
SELECT
    Order_Status,
    COUNT(*) AS Total_Orders,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM food_delivery_orders),
        2
    ) AS Percentage
FROM food_delivery_orders
GROUP BY Order_Status;
"""
result = pd.read_sql(query, engine)
result

,Order_Status,Total_Orders,Percentage
0,Delivered,84964,84.96
1,Cancelled,15036,15.04
